# Memory Efficiency: PagedAttention KV Cache

At the heart of any LLM serving system lies a critical bottleneck: memory. In this notebook, you'll discover how vLLM solves the memory management puzzle with its flagship innovation, PagedAttention. By the end of this notebook, you will be able to explain the core principles of PagedAttention for KV cache management and understand how its block-based allocation enables massive throughput and memory efficiency.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as mpatches

# Helper function for visualization
def visualize_cache(manager, title="KV Cache Physical Blocks"):
    """Visualizes the allocation of physical blocks in the KV cache."""
    num_blocks = manager.num_total_blocks
    # Create a mapping from block number to a request_id
    block_to_req = {block: 0 for block in range(num_blocks)} # 0 for free
    
    # Use unique integers for each request for coloring
    all_req_ids = sorted(list(manager.block_tables.keys()))
    req_to_color_idx = {req_id: i + 1 for i, req_id in enumerate(all_req_ids)}

    for req_id, block_table in manager.block_tables.items():
        for block_idx in block_table:
            block_to_req[block_idx] = req_to_color_idx[req_id]

    # Reshape into a grid, aiming for a squarish layout
    cols = int(np.ceil(np.sqrt(num_blocks)))
    rows = int(np.ceil(num_blocks / cols))
    grid = np.zeros(rows * cols, dtype=int)
    
    # Populate the grid with allocation data
    indices = np.array(list(block_to_req.keys()))
    values = np.array(list(block_to_req.values()))
    grid[indices] = values
    grid = grid.reshape((rows, cols))

    # Create the plot
    fig, ax = plt.subplots(figsize=(10, 5))
    cmap = plt.cm.get_cmap('viridis', len(all_req_ids) + 1)
    cax = ax.matshow(grid, cmap=cmap)
    
    # Add block numbers to the grid
    for r in range(rows):
        for c in range(cols):
            block_num = r * cols + c
            if block_num < num_blocks:
                ax.text(c, r, str(block_num), va='center', ha='center', color='white')

    # Create a legend
    patches = [mpatches.Patch(color=cmap(0), label='Free')]
    for req_id, color_idx in req_to_color_idx.items():
        patches.append(mpatches.Patch(color=cmap(color_idx), label=f'{req_id}'))
    
    plt.legend(handles=patches, bbox_to_anchor=(1.15, 1), loc='upper left')
    plt.title(title)
    ax.axis('off')
    plt.show()

### The Problem: Memory Fragmentation

Imagine you're managing a parking garage. Cars (requests) of different lengths (sequence lengths) need parking spots (memory). The traditional approach is to find a single, continuous stretch of empty spots long enough for the entire car.

This works fine at first. But what happens when cars start leaving? A long car leaves, creating a big empty stretch. Then two small cars park in that stretch, leaving a small, unusable spot between them. This is **fragmentation**. Soon, your garage is full of small, unusable gaps, and you have to turn away a new long car even though you have enough *total* empty spots.

This is exactly what happens with the KV cache in GPU memory. Each request's KV cache needs to be stored. If we allocate one big, contiguous chunk of memory for each request, we quickly run into fragmentation. A lot of memory is wasted in small, unusable gaps, limiting the number of requests we can serve concurrently and thus lowering throughput.

### The Solution: PagedAttention

What if we could park a long car in several *non-continuous* spots? This is the core idea behind PagedAttention, and it's a brilliant analogy to a concept from operating systems: **virtual memory and paging**.

Instead of viewing the GPU memory as one long road, PagedAttention breaks it into small, fixed-size chunks called **blocks**. A single request's KV cache, which logically needs to be contiguous, can now be stored in physical blocks that are scattered all over the GPU memory.

*   **Physical Blocks:** The actual, fixed-size memory chunks in the GPU.
*   **Logical Blocks:** The sequence of blocks as seen by a single request. The request thinks its memory is one continuous block sequence.
*   **Block Table:** A lookup table, unique to each request, that maps the request's logical blocks to their actual locations in physical memory. It's like a set of directions: "Your first chunk is in spot #17, your second is in spot #5, your third is in spot #42..."

This approach completely eliminates memory fragmentation and allows the system to use nearly 100% of the available memory, leading to much higher batch sizes and throughput.

In [ ]:
class PagedKVCacheManager:
    """A simplified KV cache manager using the PagedAttention concept."""

    def __init__(self, num_total_blocks: int, block_size: int):
        self.num_total_blocks = num_total_blocks
        self.block_size = block_size
        
        # A list of available physical block indices
        self.free_blocks = list(range(num_total_blocks))
        
        # Maps a request_id to its "block table" (a list of physical block indices)
        self.block_tables = {}
        
        # Tracks how many tokens are stored in the last block of each request
        self.tokens_in_last_block = {}

    def allocate_new_block(self, request_id: str):
        """Allocates a new physical block to a request."""
        if not self.free_blocks:
            raise ValueError("Out of memory: No free blocks available.")
        
        block_idx = self.free_blocks.pop(0)
        self.block_tables[request_id].append(block_idx)
        self.tokens_in_last_block[request_id] = 0
        print(f"Allocated block {block_idx} to {request_id}. Free blocks: {len(self.free_blocks)}")

    def append_token(self, request_id: str):
        """Simulates adding a token to a request's KV cache."""
        if request_id not in self.block_tables:
            # First token for this request, create its block table
            self.block_tables[request_id] = []
            self.allocate_new_block(request_id)

        # If the last block is full, allocate a new one
        if self.tokens_in_last_block[request_id] >= self.block_size:
            self.allocate_new_block(request_id)

        self.tokens_in_last_block[request_id] += 1

    def free_request(self, request_id: str):
        """Frees all blocks associated with a finished request."""
        if request_id not in self.block_tables:
            return
        
        blocks_to_free = self.block_tables.pop(request_id)
        self.free_blocks.extend(blocks_to_free)
        self.tokens_in_last_block.pop(request_id)
        print(f"Freed {request_id}. Released blocks: {blocks_to_free}. Free blocks: {len(self.free_blocks)}")

### Walking Through the Implementation

Our `PagedKVCacheManager` is a minimal model of the real system. Let's break it down.

1.  **`__init__(self, num_total_blocks, block_size)`**:
    *   `num_total_blocks`: The total number of fixed-size physical blocks our GPU memory has (e.g., 16).
    *   `block_size`: The number of tokens' KV pairs that can fit in a single block (e.g., 4).
    *   `self.free_blocks`: This is our pool of available resources. We initialize it as a list containing the indices of all blocks, from `0` to `num_total_blocks - 1`.
    *   `self.block_tables`: This is the heart of the paging system. It's a dictionary where keys are `request_id`s and values are lists of integers. Each list is the "page table" for that request, telling us which physical blocks it's using and in what order.
    *   `self.tokens_in_last_block`: A helper to know when a request's current block is full and needs a new one.

2.  **`allocate_new_block(self, request_id)`**:
    *   This function handles the core allocation logic. It checks if there are any `free_blocks`.
    *   If so, it takes one from the pool (`self.free_blocks.pop(0)`).
    *   It then appends this physical block's index to the block table of the given `request_id`.
    *   Finally, it resets the token count for the (new) last block to 0.

3.  **`append_token(self, request_id)`**:
    *   This simulates the generation of a new token for a request.
    *   It first checks if the request is new. If so, it creates an empty block table for it and immediately calls `allocate_new_block`.
    *   The crucial logic is here: `if self.tokens_in_last_block[request_id] >= self.block_size:`. If the last block assigned to the request is full, it simply allocates a *new* one. This new block can be any free block in the system; it doesn't have to be adjacent to the previous one.
    *   It then increments the token count for the last block.

4.  **`free_request(self, request_id)`**:
    *   When a request is finished, this method is called.
    *   It finds the request's block table in `self.block_tables`.
    *   It takes that list of block indices and adds them all back to the `self.free_blocks` pool, making them available for new requests.

In [ ]:
# --- Demonstration ---

# Setup a cache manager with 16 total blocks, each able to hold 4 tokens' KV cache.
cache_manager = PagedKVCacheManager(num_total_blocks=16, block_size=4)
print("--- Initial State ---")
visualize_cache(cache_manager, "Initial Empty Cache")

# 1. Request A (long prompt) arrives and needs 5 tokens.
print("\n--- Request A (long prompt) arrives ---")
for _ in range(5):
    cache_manager.append_token("Request_A")
print("Block table for Request_A:", cache_manager.block_tables["Request_A"])
visualize_cache(cache_manager, "After Request A Arrives")

# 2. Request B (short prompt) arrives and needs 3 tokens.
print("\n--- Request B (short prompt) arrives ---")
for _ in range(3):
    cache_manager.append_token("Request_B")
print("Block table for Request_B:", cache_manager.block_tables["Request_B"])
visualize_cache(cache_manager, "After Request B Arrives")

# 3. Request A generates 2 more tokens, filling its second block.
print("\n--- Request A generates 2 more tokens ---")
for _ in range(2):
    cache_manager.append_token("Request_A")
print("Block table for Request_A:", cache_manager.block_tables["Request_A"])
visualize_cache(cache_manager, "After Request A Generates More")

# 4. Request B finishes. Its blocks are freed.
print("\n--- Request B finishes ---")
cache_manager.free_request("Request_B")
visualize_cache(cache_manager, "After Request B Finishes")

# 5. Request C arrives. It can now use the blocks freed by B.
print("\n--- Request C arrives ---")
for _ in range(6):
    cache_manager.append_token("Request_C")
print("Block table for Request_C:", cache_manager.block_tables["Request_C"])
print("Notice how Request C uses block 2, which was previously used by Request B.")
visualize_cache(cache_manager, "Final State: Request C uses freed blocks")

### Going Deeper: Copy-on-Write for Parallel Outputs

PagedAttention enables another powerful optimization that is critical for features like beam search or parallel sampling, where you generate multiple possible output sequences from a single prompt.

All these parallel sequences share the same initial prompt. With a traditional memory allocator, you would have to duplicate the entire KV cache for the prompt for each parallel sequence. This is incredibly wasteful.

With PagedAttention, you can implement **Copy-on-Write (CoW)**. Instead of copying the physical data blocks, you can just copy the *block table*. All the new sequences can share the exact same physical memory blocks for the shared prompt part. When one of these sequences generates a new token, it gets its own new block, leaving the shared blocks untouched. This saves enormous amounts of memory and avoids expensive copy operations.

Let's add a `fork_request` method to our manager to see this in action.

In [ ]:
class PagedKVCacheManagerWithCoW(PagedKVCacheManager):
    """Extends the manager to support Copy-on-Write."""
    
    def fork_request(self, parent_id: str, child_id: str):
        """Creates a new request that shares the same blocks as the parent."""
        print(f"\nForking {parent_id} into {child_id} using Copy-on-Write.")
        
        # Simply copy the block table. No new memory is allocated!
        parent_block_table = self.block_tables[parent_id]
        self.block_tables[child_id] = parent_block_table.copy()
        
        # Copy the token count as well
        self.tokens_in_last_block[child_id] = self.tokens_in_last_block[parent_id]
        
        print(f"Block table for {child_id}: {self.block_tables[child_id]} (shared with parent)")

# --- Demonstration of Copy-on-Write ---
cow_cache_manager = PagedKVCacheManagerWithCoW(num_total_blocks=16, block_size=4)

# A prompt arrives
print("--- A base request arrives ---")
for _ in range(5):
    cow_cache_manager.append_token("Prompt_Req")
print("Block table for Prompt_Req:", cow_cache_manager.block_tables["Prompt_Req"])
visualize_cache(cow_cache_manager, "Base Prompt Loaded")

# Now, we want to generate two different sequences from this prompt.
# We fork the original request into two children.
cow_cache_manager.fork_request("Prompt_Req", "Child_Seq_1")
cow_cache_manager.fork_request("Prompt_Req", "Child_Seq_2")

# Note that no new blocks have been allocated yet.
print(f"\nFree blocks remaining: {len(cow_cache_manager.free_blocks)}")

# Now, let's append a new token to the first child sequence.
print("\n--- Generating token for Child_Seq_1 ---")
cow_cache_manager.append_token("Child_Seq_1")

print("\nBlock table for Prompt_Req: ", cow_cache_manager.block_tables["Prompt_Req"])
print("Block table for Child_Seq_1:", cow_cache_manager.block_tables["Child_Seq_1"])
print("Block table for Child_Seq_2:", cow_cache_manager.block_tables["Child_Seq_2"])

print("\nNotice Child_Seq_1 now has a new block (2), while the others don't.")
visualize_cache(cow_cache_manager, "After CoW and generation for Child 1")

### What We Built: A Model of PagedAttention

The visualization below shows the final state of our first demonstration. Look closely at `Request_A` (blue) and `Request_C` (purple).

-   **`Request_A`** occupies physical blocks `0` and `1`. They are contiguous, but only by chance.
-   **`Request_C`** occupies physical blocks `2` and `3`. Block `2` was freed by `Request_B` and immediately reused.

This is the power of paging: the logical sequence of blocks for a request is maintained by its block table, while the physical blocks can be located anywhere in memory. This eliminates fragmentation, enabling the system to pack requests tightly and maximize GPU utilization.

In [ ]:
# This cell re-runs the first demonstration to produce the final visualization.
final_state_manager = PagedKVCacheManager(num_total_blocks=16, block_size=4)

# 1. Request A
for _ in range(5):
    final_state_manager.append_token("Request_A")
# 2. Request B
for _ in range(3):
    final_state_manager.append_token("Request_B")
# 3. Request A generates more
for _ in range(2):
    final_state_manager.append_token("Request_A")
# 4. Request B finishes
final_state_manager.free_request("Request_B")
# 5. Request C arrives
for _ in range(6):
    final_state_manager.append_token("Request_C")
    
# Visualize the final memory layout
visualize_cache(final_state_manager, "Final Cache Layout Showing Non-Contiguous Allocation")
print("Block table for Request_A:", final_state_manager.block_tables["Request_A"])
print("Block table for Request_C:", final_state_manager.block_tables["Request_C"])
print("\nNote: Request_C re-used block 2, which was freed by Request_B.")

### Summary & Key Takeaways

In this notebook, we explored PagedAttention, vLLM's key innovation for memory management. By abstracting physical memory into a pool of blocks and giving each request a "virtual" view of its KV cache via a block table, the system achieves near-perfect memory utilization.

**Key Takeaways:**
*   **Virtual Memory Analogy:** PagedAttention works like virtual memory in an OS, mapping logical contiguous memory for a process (request) to fragmented physical memory pages (blocks).
*   **Eliminates Fragmentation:** By allocating memory in small, fixed-size blocks, PagedAttention avoids both internal and external fragmentation, allowing for much higher batch sizes.
*   **Enables Copy-on-Write:** Sharing physical blocks for common prefixes (e.g., in beam search) becomes trivial and efficient, dramatically reducing memory overhead for complex sampling strategies.

PagedAttention is the foundational technology that allows the vLLM `Scheduler` and `Executor` to work so efficiently, enabling the high-throughput performance for which the system is known. This concludes our deep dive into the core components of vLLM.